# Saved-Artifact Analysis Run

This notebook wraps the analysis scripts that summarize saved validation and motorneuron graph artifacts. Run it after the static/dynamic validation outputs and motorneuron graph pickle outputs exist.

By default the commands are only printed. Set `RUN_ANALYSIS = True` to execute the selected steps.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "calcium_transient_rising_flank").is_dir():
            return candidate
        nested = candidate / "calcium-transient-rising-flank"
        if (nested / "src" / "calcium_transient_rising_flank").is_dir():
            return nested
    raise RuntimeError("Run from the repository, package root, or notebooks directory.")


PROJECT_ROOT = find_project_root()
PYTHON = sys.executable
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
def command_text(command: list[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)


def run_steps(steps: list[tuple[str, list[str]]], *, execute: bool) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT / "src")
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
    env.setdefault("XDG_CACHE_HOME", "/tmp/font-cache")
    for index, (name, command) in enumerate(steps, start=1):
        print(f"{index}. {name}")
        print(command_text(command))
        if execute:
            subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)
    if not execute:
        print("Dry run only. Set RUN_ANALYSIS = True to execute.")

In [ ]:
RUN_ANALYSIS = False

RUN_CHEN_COMPARISON = True
RUN_EMPIRICAL_PAIRING = True
RUN_GRAPH_STABILITY = True
RUN_FALSE_POSITIVE_TRADEOFFS = True
RUN_READINESS_REPORT = True
RUN_EVIDENCE_PACKAGE = True
RUN_TODO_AUDIT = True

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
MOTORNEURON_OUTPUT_DIR = OUTPUT_ROOT / "motorneurons"
VALIDATION_RESULTS_DIR = OUTPUT_ROOT / "validation_results"

In [ ]:
steps = []
if RUN_CHEN_COMPARISON:
    steps.append(
        (
            "Chen and saved graph comparison",
            [PYTHON, "examples/build_chen_comparison_table.py", "--input-dir", str(MOTORNEURON_OUTPUT_DIR), "--output-dir", str(OUTPUT_ROOT / "chen_comparison")],
        )
    )
if RUN_EMPIRICAL_PAIRING:
    steps.append(
        (
            "paired rise-minus-fall empirical tests",
            [PYTHON, "examples/analyze_empirical_pairing.py", "--input", str(OUTPUT_ROOT / "chen_comparison" / "chen_comparison_rows.csv"), "--output-dir", str(OUTPUT_ROOT / "empirical_stats")],
        )
    )
if RUN_GRAPH_STABILITY:
    steps.append(
        (
            "saved graph support and stability",
            [PYTHON, "examples/analyze_graph_stability.py", "--input-dir", str(MOTORNEURON_OUTPUT_DIR), "--output-dir", str(OUTPUT_ROOT / "graph_stability")],
        )
    )
if RUN_FALSE_POSITIVE_TRADEOFFS:
    steps.append(
        (
            "synthetic recall and false-positive tradeoffs",
            [PYTHON, "examples/analyze_false_positive_tradeoffs.py", "--input-dir", str(VALIDATION_RESULTS_DIR), "--output-dir", str(OUTPUT_ROOT / "validation_tradeoffs")],
        )
    )
if RUN_READINESS_REPORT:
    steps.append(
        (
            "result readiness report",
            [PYTHON, "examples/build_result_readiness_report.py", "--output-root", str(OUTPUT_ROOT), "--output-dir", str(OUTPUT_ROOT / "result_readiness")],
        )
    )
if RUN_EVIDENCE_PACKAGE:
    steps.append(
        (
            "manuscript evidence package",
            [PYTHON, "examples/build_manuscript_evidence_package.py", "--output-root", str(OUTPUT_ROOT), "--output-dir", str(OUTPUT_ROOT / "manuscript_evidence")],
        )
    )
if RUN_TODO_AUDIT:
    steps.append(
        (
            "todo completion audit",
            [PYTHON, "examples/build_todo_completion_audit.py", "--output-dir", str(OUTPUT_ROOT / "todo_completion")],
        )
    )

run_steps(steps, execute=RUN_ANALYSIS)

In [ ]:
readiness_path = OUTPUT_ROOT / "result_readiness" / "summary.json"
if readiness_path.is_file():
    print(json.dumps(json.loads(readiness_path.read_text()), indent=2))
else:
    print(f"No readiness summary yet at {readiness_path}")